In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv("IEA-EV-dataEV salesHistoricalCars.csv")

df.head()

,region,category,parameter,mode,powertrain,year,unit,value
0,Australia,Historical,EV sales,Cars,BEV,2011,Vehicles,49.00000
1,Australia,Historical,EV stock share,Cars,EV,2011,percent,0.00039
2,Australia,Historical,EV sales share,Cars,EV,2011,percent,0.00650
3,Australia,Historical,EV stock,Cars,BEV,2011,Vehicles,49.00000
4,Australia,Historical,EV stock,Cars,BEV,2012,Vehicles,220.00000


In [31]:
df_clean = df[["region", "year", "parameter", "value"]].copy()

df_clean.head()

,region,year,parameter,value
0,Australia,2011,EV sales,49.00000
1,Australia,2011,EV stock share,0.00039
2,Australia,2011,EV sales share,0.00650
3,Australia,2011,EV stock,49.00000
4,Australia,2012,EV stock,220.00000


In [35]:
# Keep the EV indicators most relevant to the project
selected_parameters = [
    "EV sales",
    "EV stock",
    "EV sales share",
    "EV stock share"
]

df_clean = df_clean[df_clean["parameter"].isin(selected_parameters)].copy()

df_clean.head()

,region,year,parameter,value
0,Australia,2011,EV sales,49.00000
1,Australia,2011,EV stock share,0.00039
2,Australia,2011,EV sales share,0.00650
3,Australia,2011,EV stock,49.00000
4,Australia,2012,EV stock,220.00000


In [41]:
# Reshape the dataset from long format to wide format
df_wide = df_clean.pivot_table(
    index=["region", "year"],
    columns="parameter",
    values="value"
).reset_index()

df_wide.head()

parameter,region,year,EV sales,EV sales share,EV stock,EV stock share
0,Australia,2011,49.0,0.0065,49.0,0.00039
1,Australia,2012,125.0,0.0300,150.0,0.00240
2,Australia,2013,145.0,0.0340,295.0,0.00460
3,Australia,2014,660.0,0.1600,940.0,0.01400
4,Australia,2015,880.0,0.2000,1800.0,0.02700


In [43]:
# Rename columns for easier use in modeling
df_wide = df_wide.rename(columns={
    "EV sales": "EV_sales",
    "EV stock": "EV_stock",
    "EV sales share": "sales_share",
    "EV stock share": "stock_share"
})

df_wide.head()

parameter,region,year,EV_sales,sales_share,EV_stock,stock_share
0,Australia,2011,49.0,0.0065,49.0,0.00039
1,Australia,2012,125.0,0.0300,150.0,0.00240
2,Australia,2013,145.0,0.0340,295.0,0.00460
3,Australia,2014,660.0,0.1600,940.0,0.01400
4,Australia,2015,880.0,0.2000,1800.0,0.02700


In [47]:
# dropping missing EV_sales for the analysis of time series
df_wide = df_wide.dropna(subset=["EV_sales"]).copy()

print("Shape after dropping missing EV_sales:", df_wide.shape)
df_wide.head()

Shape after dropping missing EV_sales: (606, 6)


parameter,region,year,EV_sales,sales_share,EV_stock,stock_share
0,Australia,2011,49.0,0.0065,49.0,0.00039
1,Australia,2012,125.0,0.0300,150.0,0.00240
2,Australia,2013,145.0,0.0340,295.0,0.00460
3,Australia,2014,660.0,0.1600,940.0,0.01400
4,Australia,2015,880.0,0.2000,1800.0,0.02700


In [49]:
# Remove the column index name left over from pivot_table
df_wide.columns.name = None

df_wide.head()

,region,year,EV_sales,sales_share,EV_stock,stock_share
0,Australia,2011,49.0,0.0065,49.0,0.00039
1,Australia,2012,125.0,0.0300,150.0,0.00240
2,Australia,2013,145.0,0.0340,295.0,0.00460
3,Australia,2014,660.0,0.1600,940.0,0.01400
4,Australia,2015,880.0,0.2000,1800.0,0.02700


In [53]:
# Save cleaned dataset forthe analysis of time series
df_wide.to_csv("ev_cleaned_for_time_series.csv", index=False)

In [55]:
# Create dataset for Bayesian regression
df_bayes = df_wide.copy()

print("Original shape:", df_bayes.shape)

Original shape: (606, 6)


In [57]:
# Drop rows with any missing values
df_bayes = df_bayes.dropna().copy()

print("Shape after removing missing values:", df_bayes.shape)

df_bayes.head()

Shape after removing missing values: (458, 6)


,region,year,EV_sales,sales_share,EV_stock,stock_share
0,Australia,2011,49.0,0.0065,49.0,0.00039
1,Australia,2012,125.0,0.0300,150.0,0.00240
2,Australia,2013,145.0,0.0340,295.0,0.00460
3,Australia,2014,660.0,0.1600,940.0,0.01400
4,Australia,2015,880.0,0.2000,1800.0,0.02700


In [61]:
# Save dataset for the analysis of Bayesian regression
df_bayes.to_csv("ev_cleaned_for_bayesian.csv", index=False)